# Plot And Filter Weekly HQ Series

This notebook:

1. Loads the weekly values and datetime CSVs.
2. Displays a zero-based series index table so you can identify columns to omit.
3. Plots every time series in paginated subplot grids.
4. Saves each plot page as both `.pgf` and `.pdf`.
5. Lets you enter the zero-based indices to omit and saves filtered output CSVs.

The omit step uses column indices, not row indices.

In [ ]:
from pathlib import Path
import math

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.dates import AutoDateLocator, ConciseDateFormatter
import pandas as pd
from IPython.display import display

SINGLE_COL = (3.5, 2.5)
DOUBLE_COL = (6.5, 3.5)

mpl.rcParams.update({
    'pgf.texsystem': 'pdflatex',
    'font.family': 'serif',
    'text.usetex': True,
    'pgf.rcfonts': False,
    'pgf.preamble': r'\usepackage{amsfonts}\usepackage{amssymb}\usepackage{amsmath}',
    'lines.linewidth': 1,
    'figure.figsize': SINGLE_COL,
    'font.size': 9,
    'savefig.dpi': 300,
})

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

In [ ]:
# DATA_VALUES_REL = Path('data/hq/ts_weekly_values_edited.csv')
# DATA_DATETIME_REL = Path('data/hq/ts_weekly_datetimes_edited.csv')
DATA_VALUES_REL = Path('data/hq/ts_weekly_values_final.csv')
DATA_DATETIME_REL = Path('data/hq/ts_weekly_datetimes_final.csv')
OUT_VALUES_REL = Path('data/hq/ts_weekly_values_final2.csv')
OUT_DATETIME_REL = Path('data/hq/ts_weekly_datetimes_final2.csv')
PLOT_EXPORT_DIR_REL = Path('experiments/out/plot_and_filter_weekly_series')
EXPORT_PLOTS = True


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *list(cwd.parents)[:4]]
    for base in candidates:
        if (base / 'data' / 'hq').exists():
            return base
    raise FileNotFoundError('Could not locate the repo root containing data/hq from the current working directory.')


REPO_ROOT = find_repo_root()
VALUES_CSV = (REPO_ROOT / DATA_VALUES_REL).resolve()
DATETIME_CSV = (REPO_ROOT / DATA_DATETIME_REL).resolve()
OUT_VALUES_CSV = (REPO_ROOT / OUT_VALUES_REL).resolve()
OUT_DATETIME_CSV = (REPO_ROOT / OUT_DATETIME_REL).resolve()
PLOT_EXPORT_DIR = (REPO_ROOT / PLOT_EXPORT_DIR_REL).resolve()

print('Repo root      :', REPO_ROOT)
print('Input values   :', VALUES_CSV)
print('Input dates    :', DATETIME_CSV)
print('Output values  :', OUT_VALUES_CSV)
print('Output dates   :', OUT_DATETIME_CSV)
print('Plot export dir:', PLOT_EXPORT_DIR)

In [ ]:
def load_csv_pair(values_csv: Path, datetime_csv: Path):
    values_df = pd.read_csv(values_csv)
    dates_df = pd.read_csv(datetime_csv)

    if values_df.shape != dates_df.shape:
        raise ValueError(f'Shape mismatch: values={values_df.shape}, dates={dates_df.shape}')
    if list(values_df.columns) != list(dates_df.columns):
        raise ValueError('Column mismatch between values and datetimes CSVs.')

    return values_df, dates_df


def tex_escape(text: str) -> str:
    replacements = {
        '\\': r'\textbackslash{}',
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
    }
    return ''.join(replacements.get(char, char) for char in str(text))


def series_frame(values_df: pd.DataFrame, dates_df: pd.DataFrame, series_idx: int) -> pd.DataFrame:
    frame = pd.DataFrame({
        'date': pd.to_datetime(dates_df.iloc[:, series_idx], errors='coerce'),
        'value': pd.to_numeric(values_df.iloc[:, series_idx], errors='coerce'),
    })
    frame = frame.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
    return frame


def build_series_index(values_df: pd.DataFrame, dates_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for series_idx, name in enumerate(values_df.columns):
        frame = series_frame(values_df, dates_df, series_idx)
        non_null_values = int(frame['value'].notna().sum())
        first_date = frame['date'].min()
        last_date = frame['date'].max()
        rows.append({
            'series_index': series_idx,
            'series_name': name,
            'non_null_values': non_null_values,
            'first_date': first_date.date().isoformat() if pd.notna(first_date) else None,
            'last_date': last_date.date().isoformat() if pd.notna(last_date) else None,
        })
    return pd.DataFrame(rows)


def parse_omit_spec(spec: str, n_series: int) -> list[int]:
    spec = spec.replace('\n', ',').replace(';', ',').strip()
    if not spec:
        return []

    tokens = []
    for chunk in spec.split(','):
        chunk = chunk.strip()
        if not chunk:
            continue
        tokens.extend(chunk.split())

    indices = sorted({int(token) for token in tokens})
    invalid = [idx for idx in indices if idx < 0 or idx >= n_series]
    if invalid:
        raise IndexError(f'Invalid series indices: {invalid}. Valid range is 0 to {n_series - 1}.')
    return indices


def drop_series_by_index(values_df: pd.DataFrame, dates_df: pd.DataFrame, omit_indices: list[int]):
    omit_set = set(omit_indices)
    keep_mask = [idx not in omit_set for idx in range(values_df.shape[1])]
    return values_df.loc[:, keep_mask].copy(), dates_df.loc[:, keep_mask].copy()


def save_figure_pair(fig, export_dir: Path, stem: str):
    export_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(export_dir / f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(export_dir / f'{stem}.pgf', bbox_inches='tight')

In [ ]:
values_df, dates_df = load_csv_pair(VALUES_CSV, DATETIME_CSV)
series_index_df = build_series_index(values_df, dates_df)

print(f'Loaded {values_df.shape[1]} series and {values_df.shape[0]} weekly rows.')
display(series_index_df)

In [ ]:
def plot_all_series(
    values_df: pd.DataFrame,
    dates_df: pd.DataFrame,
    ncols: int = 2,
    series_per_page: int = 6,
    export_plots: bool = EXPORT_PLOTS,
    export_dir: Path = PLOT_EXPORT_DIR,
):
    total_series = values_df.shape[1]

    for page_idx, start in enumerate(range(0, total_series, series_per_page)):
        stop = min(start + series_per_page, total_series)
        indices = list(range(start, stop))
        n_plots = len(indices)
        nrows = math.ceil(n_plots / ncols)

        fig, axes = plt.subplots(
            nrows=nrows,
            ncols=ncols,
            figsize=(DOUBLE_COL[0], SINGLE_COL[1] * nrows),
            squeeze=False,
        )
        axes_flat = axes.ravel()

        for plot_idx, (ax, series_idx) in enumerate(zip(axes_flat, indices)):
            frame = series_frame(values_df, dates_df, series_idx)
            name = values_df.columns[series_idx]

            ax.plot(frame['date'], frame['value'], color='tab:red')
            ax.text(
                0.01,
                0.98,
                f'[{series_idx}] {tex_escape(name)}',
                transform=ax.transAxes,
                ha='left',
                va='top',
                fontsize=8,
            )

            locator = AutoDateLocator()
            ax.xaxis.set_major_locator(locator)
            ax.xaxis.set_major_formatter(ConciseDateFormatter(locator))
            ax.tick_params(axis='both', labelsize=8)

            if plot_idx // ncols == nrows - 1:
                ax.set_xlabel('date')
            else:
                ax.set_xlabel('')

            if plot_idx % ncols == 0:
                ax.set_ylabel('value')
            else:
                ax.set_ylabel('')

            ax.margins(x=0.01)

        for ax in axes_flat[n_plots:]:
            ax.axis('off')

        fig.tight_layout()

        if export_plots:
            save_figure_pair(fig, export_dir, f'weekly_series_page_{page_idx:03d}')

        plt.show()


plot_all_series(values_df, dates_df)

## Omit And Save

Edit `OMIT_SPEC` below with the zero-based series indices you want to omit, then run that cell and the final save cell.

Examples:
- `"3"`
- `"3, 18, 42"`
- `"3 18 42"`

In [ ]:
# OMIT_SPEC = ''  # Example: '3, 18, 42'

# OMIT_INDICES = parse_omit_spec(OMIT_SPEC, n_series=values_df.shape[1])
# omitted_series_df = series_index_df[series_index_df['series_index'].isin(OMIT_INDICES)].reset_index(drop=True)

# print('Omitting indices:', OMIT_INDICES if OMIT_INDICES else 'none')
# display(omitted_series_df if not omitted_series_df.empty else pd.DataFrame(columns=series_index_df.columns))

In [ ]:
# filtered_values_df, filtered_dates_df = drop_series_by_index(values_df, dates_df, OMIT_INDICES)

# filtered_values_df.to_csv(OUT_VALUES_CSV, index=False)
# filtered_dates_df.to_csv(OUT_DATETIME_CSV, index=False)

# print(f'Saved values to: {OUT_VALUES_CSV}')
# print(f'Saved dates to : {OUT_DATETIME_CSV}')
# print(f'Removed {len(OMIT_INDICES)} series. New shape: {filtered_values_df.shape}')